
# BERT-Based Sentiment Analysis Using a Pre-Trained Model

## Deep Learning Practical

**Objective:**  
To implement a pre-trained BERT model for sentiment analysis using a sample dataset, fine-tune the model, evaluate its performance, and classify custom text reviews as positive or negative.

### Technologies Used
- Python
- Google Colab
- PyTorch
- Hugging Face Transformers
- Hugging Face Datasets
- Scikit-learn
- Matplotlib
- Seaborn

### Model
`bert-base-uncased`

### Dataset
IMDB Movie Review Dataset

### Classification
- `0` → Negative
- `1` → Positive



## 1. Introduction

BERT stands for **Bidirectional Encoder Representations from Transformers**. It is a Transformer-based pre-trained language model that learns contextual representations of words from large text corpora.

In this practical, a pre-trained BERT model is fine-tuned for **binary sentiment classification**. The model receives movie reviews from the IMDB dataset and predicts whether each review is positive or negative.

### Overall Workflow

```text
IMDB Dataset
      ↓
Data Loading
      ↓
Data Sampling
      ↓
BERT Tokenization
      ↓
Pre-trained BERT
      ↓
Fine-Tuning
      ↓
Evaluation
      ↓
Confusion Matrix and Classification Report
      ↓
Custom Text Prediction
```


## 2. Install Required Libraries

Run this cell first in Google Colab.

In [ ]:
!pip install -q -U transformers datasets evaluate accelerate scikit-learn seaborn

## 3. Import Required Libraries

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

import evaluate

print("Libraries imported successfully.")


## 4. Check GPU Availability

BERT fine-tuning is faster when a GPU is available. In Google Colab, select **Runtime → Change runtime type → T4 GPU**.

In [ ]:

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")


## 5. Load the IMDB Dataset

The IMDB dataset contains movie reviews with binary sentiment labels. The current Hugging Face repository identifier is `stanfordnlp/imdb`.

In [ ]:

dataset = load_dataset("stanfordnlp/imdb")

print(dataset)


## 6. Inspect a Sample Review

In [ ]:

print("Sample Review:")
print(dataset["train"][0]["text"])

print("\nLabel:")
print(dataset["train"][0]["label"])

print("\nLabel Meaning:")
print("0 = Negative")
print("1 = Positive")



## 7. Create a Smaller Dataset

The original IMDB dataset contains many reviews. To make the practical faster, we use:

- **5,000 training reviews**
- **1,000 testing reviews**

A fixed random seed is used so that the selected samples are reproducible.


In [ ]:

train_dataset = dataset["train"].shuffle(seed=42).select(range(5000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(1000))

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))



## 8. Load the Pre-Trained BERT Tokenizer

The tokenizer converts text into numerical tokens that can be processed by BERT.

We use:

`bert-base-uncased`

The model is uncased, meaning that it does not distinguish between uppercase and lowercase letters.


In [ ]:

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully.")


## 9. Demonstrate BERT Tokenization

In [ ]:

text = "I loved this movie."

tokens = tokenizer.tokenize(text)

print("Original text:")
print(text)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(tokenizer.convert_tokens_to_ids(tokens))



## 10. Tokenize the Dataset

Each review is converted into:
- `input_ids` — numerical token IDs
- `attention_mask` — identifies actual tokens and padding
- `token_type_ids` — segment information used by BERT when applicable

Reviews are truncated or padded to a maximum length of 128 tokens.


In [ ]:

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenization completed.")


## 11. Inspect Tokenized Data

In [ ]:
print(train_tokenized[0])


## 12. Load the Pre-Trained BERT Classification Model

`AutoModelForSequenceClassification` adds a classification head on top of BERT.

Since this is binary sentiment classification:

- `0` → Negative
- `1` → Positive
- `num_labels = 2`


In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("BERT classification model loaded successfully.")


## 13. Move the Model to GPU or CPU

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print("Using device:", device)


## 14. Define Evaluation Metric

Accuracy is used as the main evaluation metric.

In [ ]:

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )



## 15. Define Training Arguments

Important parameters:
- Learning rate: `2e-5`
- Batch size: `8`
- Epochs: `2`
- Weight decay: `0.01`

These settings provide a practical balance between training time and model performance.


In [ ]:

training_args = TrainingArguments(
    output_dir="./bert_sentiment_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_strategy="steps",
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none"
)

print("Training arguments configured.")


## 16. Create the Trainer

In [ ]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer created successfully.")



## 17. Fine-Tune the BERT Model

This is the main training step. The pre-trained BERT model is fine-tuned using the labeled IMDB reviews.


In [ ]:
train_result = trainer.train()

## 18. Evaluate the Fine-Tuned Model

In [ ]:

results = trainer.evaluate()

print("Evaluation Results:")
for key, value in results.items():
    print(f"{key}: {value}")


## 19. Display Training History

In [ ]:

history = trainer.state.log_history
history_df = pd.DataFrame(history)

display(history_df)


## 20. Plot Training Loss

In [ ]:

train_loss = history_df[history_df["loss"].notna()]

plt.figure(figsize=(8, 5))
plt.plot(train_loss["step"], train_loss["loss"], marker="o")
plt.xlabel("Training Steps")
plt.ylabel("Training Loss")
plt.title("BERT Training Loss")
plt.grid(True)
plt.show()


## 21. Plot Validation Loss

In [ ]:

eval_loss = history_df[history_df["eval_loss"].notna()]

plt.figure(figsize=(8, 5))
plt.plot(eval_loss["epoch"], eval_loss["eval_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("BERT Validation Loss")
plt.grid(True)
plt.show()



## 22. Generate Predictions on the Test Dataset

The fine-tuned model is used to predict sentiment labels for the complete selected test set.


In [ ]:

prediction_output = trainer.predict(test_tokenized)

predicted_labels = np.argmax(
    prediction_output.predictions,
    axis=1
)

actual_labels = prediction_output.label_ids

print("Predictions generated successfully.")


## 23. Confusion Matrix

In [ ]:

cm = confusion_matrix(
    actual_labels,
    predicted_labels
)

print("Confusion Matrix:")
print(cm)


In [ ]:

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("BERT Sentiment Analysis Confusion Matrix")

plt.show()


## 24. Classification Report

In [ ]:

report = classification_report(
    actual_labels,
    predicted_labels,
    target_names=["Negative", "Positive"]
)

print(report)



## 25. Test the Model on Custom Reviews

The following examples demonstrate how the fine-tuned BERT model can classify new, unseen text.


In [ ]:

custom_reviews = [
    "This movie was absolutely fantastic. I really enjoyed it.",
    "The movie was boring and a complete waste of time.",
    "The acting was excellent and the story was very interesting.",
    "I hated this movie. It was terrible.",
    "It was an average movie, nothing special."
]

inputs = tokenizer(
    custom_reviews,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

model.eval()

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(
    outputs.logits,
    dim=1
)

for review, prediction in zip(custom_reviews, predictions):
    sentiment = "Positive" if prediction.item() == 1 else "Negative"

    print("Review:", review)
    print("Prediction:", sentiment)
    print("-" * 80)



## 26. Save the Fine-Tuned Model

The fine-tuned BERT model and tokenizer can be saved for future use.


In [ ]:

save_path = "./bert_sentiment_model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved at:", save_path)



# 27. Conclusion

A pre-trained BERT model was successfully implemented for sentiment analysis using the IMDB movie review dataset. The text data was tokenized using the BERT tokenizer, and the pre-trained `bert-base-uncased` model was fine-tuned for binary classification.

The model was evaluated using accuracy, loss, a confusion matrix, precision, recall, and F1-score. Finally, custom movie reviews were supplied to demonstrate sentiment prediction on unseen text.

### Key Learning Outcomes

- Understanding pre-trained Transformer models
- Understanding BERT tokenization
- Fine-tuning BERT for text classification
- Evaluating a text classification model
- Using confusion matrices and classification reports
- Performing predictions on custom text

### Result

The implementation successfully performs binary sentiment classification:

**Positive Review → Positive**

**Negative Review → Negative**

> Record the actual accuracy, loss, precision, recall, and F1-score produced by your execution in the final practical report.



# 28. Viva Questions

### Q1. What is BERT?
BERT stands for Bidirectional Encoder Representations from Transformers. It is a pre-trained Transformer-based language model.

### Q2. Did you train BERT from scratch?
No. A pre-trained `bert-base-uncased` model was fine-tuned on the IMDB dataset.

### Q3. What is fine-tuning?
Fine-tuning means adapting a pre-trained model to a specific task using task-specific labeled data.

### Q4. What is tokenization?
Tokenization converts text into tokens and numerical IDs that can be processed by the model.

### Q5. What is an attention mask?
An attention mask tells BERT which positions contain actual tokens and which positions are padding.

### Q6. Why are there two labels?
The task is binary sentiment classification: positive and negative.

### Q7. What is the purpose of the classification layer?
It converts the representation produced by BERT into class predictions.

### Q8. Why is a pre-trained model useful?
It has already learned general language patterns from large text datasets, so it can be adapted to a specific task with comparatively less task-specific training data.
